# Barren Plateau Analysis
Run this entire notebook top to bottom. Takes ~15-25 min on Colab CPU.
At the end, download `bp_results.json` and `fig_barren_plateau.png`.

In [1]:
!pip install qiskit qiskit-aer -q

import numpy as np
import matplotlib.pyplot as plt
import math, json, time, warnings
warnings.filterwarnings('ignore')
from itertools import combinations, product
from qiskit.circuit import QuantumCircuit, ParameterVector
from qiskit.quantum_info import SparsePauliOp, Statevector
from scipy.optimize import minimize

KAPPA = 2
MU = 5.0
np.random.seed(42)
print('Setup complete.')

ERROR: Could not find a version that satisfies the requirement qiskit (from versions: none)
ERROR: No matching distribution found for qiskit


ModuleNotFoundError: No module named 'qiskit'

In [ ]:
# ── Core functions ──────────────────────────────────────────

def build_pauli_set(n_k, kappa, n_qubits):
    out = []
    for pos in combinations(range(n_qubits), kappa):
        for ops in product(['X','Y','Z'], repeat=kappa):
            arr = ['I']*n_qubits
            for p, o in zip(pos, ops):
                arr[p] = o
            out.append(''.join(reversed(arr)))
            if len(out) == n_k:
                return out
    raise ValueError(f'n_k={n_k} exceeds capacity for n_qubits={n_qubits}')

def build_hea(n_qubits, n_layers):
    theta = ParameterVector('θ', 2 * n_qubits * n_layers)
    qc = QuantumCircuit(n_qubits)
    idx = 0
    for _ in range(n_layers):
        for q in range(n_qubits):
            qc.ry(theta[idx], q); idx += 1
            qc.rz(theta[idx], q); idx += 1
        for q in range(n_qubits - 1):
            qc.cx(q, q + 1)
    return qc, theta

def get_correlators(qc, theta, params, pauli_strs):
    sv = Statevector(qc.assign_parameters(dict(zip(theta, params))))
    return np.array([
        sv.expectation_value(SparsePauliOp.from_list([(p, 1.0)])).real
        for p in pauli_strs
    ])

def make_k_loops(k):
    """k disjoint hollow triangles → beta_1 = k, n_k = 3k edges."""
    edges = []
    for i in range(k):
        b = 3 * i
        edges += [(b, b+1), (b, b+2), (b+1, b+2)]
    n_pts = 3 * k
    ne = len(edges)
    B1 = np.zeros((n_pts, ne))
    for j, (u, v) in enumerate(edges):
        B1[u, j] = -1; B1[v, j] = 1
    return B1.T @ B1

print('Functions loaded.')

In [ ]:
# ── Gradient variance estimator ─────────────────────────────

def compute_gradient_variance(n_qubits, Delta, pauli_strs,
                               n_samples=500, null_vecs=None):
    if null_vecs is None:
        null_vecs = []
    n_layers = max(1, 2 * n_qubits)
    qc, theta = build_hea(n_qubits, n_layers)
    n_params = qc.num_parameters
    eps = 1e-3

    def loss(params):
        c = get_correlators(qc, theta, params, pauli_strs)
        n2 = c @ c
        if n2 < 1e-12:
            return 1.0
        L = float(c @ Delta @ c) / n2
        nc = np.sqrt(n2)
        for cv in null_vecs:
            ov = (c @ cv) / (nc * np.linalg.norm(cv) + 1e-12)
            L += MU * ov**2
        return L

    ell = np.random.randint(n_params)
    grads = []
    for _ in range(n_samples):
        p = np.random.uniform(-np.pi, np.pi, n_params)
        p_plus = p.copy();  p_plus[ell]  += eps
        p_minus = p.copy(); p_minus[ell] -= eps
        grads.append((loss(p_plus) - loss(p_minus)) / (2 * eps))

    return float(np.var(grads))

print('Gradient variance function ready.')

In [ ]:
# ── MAIN: Barren Plateau Analysis ───────────────────────────
# BUG FIX: use k_loops = nq**2 // 3, n_k = k_loops * 3
# This avoids zero-padding which caused Var=0 at n=8 previously.

N_QUBIT_LIST      = [4, 6, 8, 10, 12]
N_DEFLATION_ROUNDS = 3     # j = 0, 1, 2
N_GRAD_SAMPLES    = 500    # full count for paper

bp_results = {}  # {n_qubits: [var_j0, var_j1, var_j2]}

print('Running barren plateau analysis...')
print(f'Config: {N_QUBIT_LIST} qubits x {N_DEFLATION_ROUNDS} rounds x {N_GRAD_SAMPLES} samples')
print()

for nq in N_QUBIT_LIST:
    # ── FIXED: no padding ──
    k_loops = nq**2 // 3
    n_k     = k_loops * 3      # exact size, no zero rows
    Delta   = make_k_loops(k_loops)
    assert Delta.shape == (n_k, n_k), f'Shape mismatch: {Delta.shape} vs ({n_k},{n_k})'

    ps = build_pauli_set(n_k, KAPPA, nq)
    vars_per_round = []

    t_start = time.perf_counter()

    # j=0: no deflation
    v0 = compute_gradient_variance(nq, Delta, ps, n_samples=N_GRAD_SAMPLES)
    vars_per_round.append(v0)
    print(f'  n={nq:2d}  n_k={n_k:4d}  j=0: Var={v0:.6f}', end='', flush=True)

    # j=1,2: simulate having found j null vectors
    null_vecs_sim = []
    for j in range(1, N_DEFLATION_ROUNDS):
        cv = np.random.randn(n_k)
        cv /= np.linalg.norm(cv)
        null_vecs_sim.append(cv)
        vj = compute_gradient_variance(nq, Delta, ps,
                                        n_samples=N_GRAD_SAMPLES,
                                        null_vecs=list(null_vecs_sim))
        vars_per_round.append(vj)
        print(f'  j={j}: Var={vj:.6f}', end='', flush=True)

    elapsed = time.perf_counter() - t_start
    bp_results[nq] = vars_per_round
    print(f'  ({elapsed:.0f}s)')

print()
print('Barren plateau analysis complete.')

with open('bp_results.json', 'w') as f:
    json.dump({str(k): v for k, v in bp_results.items()}, f, indent=2)
print('Saved bp_results.json')

In [ ]:
# ── Plot ────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(8, 5))
colors  = ['blue', 'orange', 'green']
markers = ['o', 's', '^']

for j in range(N_DEFLATION_ROUNDS):
    nqs    = sorted(bp_results.keys())
    vars_j = [bp_results[nq][j] for nq in nqs if j < len(bp_results[nq])]
    valid  = [(nq, v) for nq, v in zip(nqs, vars_j) if v > 0]
    if not valid:
        continue
    nqs_v, vars_v = zip(*valid)

    ax.semilogy(nqs_v, vars_v, color=colors[j], marker=markers[j],
                lw=2, markersize=8, label=f'j={j} deflation steps')

    log_n = np.log(nqs_v)
    log_v = np.log(vars_v)
    slope, intercept = np.polyfit(log_n, log_v, 1)
    fit_line = np.exp(intercept) * np.array(nqs_v) ** slope
    ax.semilogy(nqs_v, fit_line, color=colors[j], ls='--', alpha=0.5,
                label=f'j={j} fit: n^{slope:.2f}')
    print(f'j={j}: power-law slope = {slope:.3f}  ({"polynomial ✓" if slope > -15 else "check"})')

ax.set_xlabel('Number of qubits n')
ax.set_ylabel('Var[∂L/∂θ_ℓ]')
ax.set_title('Barren Plateau Analysis: Gradient Variance vs Qubits\n'
             '(Polynomial decay = trainable; Exponential = barren plateau)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xticks(N_QUBIT_LIST)

plt.tight_layout()
plt.savefig('fig_barren_plateau.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_barren_plateau.png')
print()
print('DONE — download bp_results.json and fig_barren_plateau.png')